### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [8]:
%load_ext autoreload
%autoreload 2
%env WANDB_API_KEY=228df7fc4aa2ceb97f82a7c6fa84f36eda50cf73

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: WANDB_API_KEY=228df7fc4aa2ceb97f82a7c6fa84f36eda50cf73


### Import packages
We'll need the `pytorch_lightning` and `nugraph` packages imported in order to train

In [9]:
import os
from pathlib import Path
import nugraph as ng
import pytorch_lightning as pl

### Set model and data to use

This allows the user to switch out different model architectures and datasets

In [10]:
Data = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

### Configure data module
Declare a data module. Depending on where you're working, you should edit the data path below to point to a valid data location.

In [11]:
nudata = Data("/sps/lbno/zappacosta/v10_11/DUNEVD_TRAINING/ProcessProva_50_5.0000.h5", batch_size=10, model=Model)

### Configure network
Declare a model. You can edit the arguments below to change the network configuration.

In [12]:
nugraph = Model(
    in_features=5,
    hit_features=128,
    nexus_features=32,
    instance_features=32,
    interaction_features=32,
    semantic_classes=nudata.semantic_classes,
    event_classes=nudata.event_classes,
    num_iters=5,
    event_head=False,
    semantic_head=True,
    filter_head=True,
    vertex_head=False,
    instance_head=True,
    use_checkpointing=True,
    lr=0.001)

### Configure logger and callbacks
Declare a tensorboard logger and define the output directory, so we can monitor network training. Also define a callback so we can monitor learning rate evolution.

In [13]:
name = "test"
#logdir = Path(os.environ["NUGRAPH_LOG"])/name
logdir = Path("/sps/lbno/zappacosta/v10_11/DUNEVD_TRAINING_LOGS/tests")
logdir.mkdir(parents=True, exist_ok=True)
logger = pl.loggers.WandbLogger(save_dir=logdir, project="nugraph3", name="test",
                                log_model="all")
callbacks = [
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
    pl.callbacks.ModelCheckpoint(monitor="loss/val", mode="min"),
]

### Declare trainer and run training
First we set the training device. To train with a GPU, pass an integer  otherwise, it defaults to CPU training. We then instantiate a PyTorch Lightning trainer that we'll use for training, and then run the training stage, which iterates over all batches in the train and validation datasets to optimise model parameters, writing output metrics to tensorboard.

In [14]:
accelerator, devices = ng.util.configure_device()
trainer = pl.Trainer(accelerator=accelerator,
                     devices=devices,
                     max_epochs=2,
                     logger=logger,
                     callbacks=callbacks)
trainer.fit(nugraph, datamodule=nudata)
trainer.test(datamodule=nudata)

/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sps/lbno/zappacosta/miniconda3/envs/numl/lib/python ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
Loading `train_dataloader` to estimate number of stepping batches.
/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (3) is smaller than 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

tensor([], size=(2209, 0))tensor([], size=(42, 0))

tensor([], size=(1, 0))
tensor([], size=(1, 0))tensor([], size=(1, 0))

tensor([], size=(3, 0))
tensor([], size=(5545, 0))
tensor([], size=(1, 0))
tensor([], size=(33, 0))
tensor([], size=(138, 0))
tensor([], size=(1, 0))
tensor([], size=(1, 0))
tensor([], size=(283, 0))
tensor([], size=(1, 0))
tensor([], size=(7, 0))
tensor([], size=(1082, 0))
tensor([], size=(1, 0))
tensor([], size=(10, 0))
tensor([], size=(3584, 0))
tensor([], size=(1, 0))
tensor([], size=(15, 0))
tensor([], size=(945, 0))
tensor([], size=(1, 0))
tensor([], size=(5, 0))
tensor([], size=(1565, 0))
tensor([], size=(1, 0))
tensor([], size=(3, 0))
tensor([], size=(2610, 0))
tensor([], size=(1, 0))
tensor([], size=(5, 0))
tensor([], size=(3393, 0))
tensor([], size=(1, 0))
tensor([], size=(8, 0))
tensor([], size=(671, 0))
tensor([], size=(6233, 0))
tensor([], size=(1, 0))
tensor([], size=(1, 0))tensor([], size=(9, 0))

tensor([], size=(29, 0))
tensor([], size=(1585, 0))


Traceback (most recent call last):
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/multiprocessing/util.py", line 367, in _run_finalizers
    finalizer()
    ~~~~~~~~~^^
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/multiprocessing/util.py", line 291, in __call__
    res = self._callback(*self._args, **self._kwargs)
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/multiprocessing/util.py", line 145, in _remove_temp_dir
    rmtree(tempdir)
    ~~~~~~^^^^^^^^^
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/shutil.py", line 763, in rmtree
    _rmtree_safe_fd(stack, onexc)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/shutil.py", line 707, in _rmtree_safe_fd
    onexc(func, path, err)
    ~~~~~^^^^^^^^^^^^^^^^^
  File "/sps/lbno/zappacosta/miniconda3/envs/numl/lib/python3.13/shutil.py", line 700, in _rmtree_safe_fd
    onexc(os.unlink, fullname, err)
    ~~~~~^^^^^^^^^

tensor([], size=(650, 0))
tensor([], size=(1, 0))
tensor([], size=(7, 0))
tensor([], size=(3519, 0))
tensor([], size=(1, 0))
tensor([], size=(4, 0))
tensor([], size=(1600, 0))
tensor([], size=(1, 0))
tensor([], size=(1, 0))
tensor([], size=(1574, 0))
tensor([], size=(1, 0))
tensor([], size=(18, 0))

RuntimeError: The size of tensor a (8) must match the size of tensor b (5) at non-singleton dimension 1


tensor([], size=(1, 0))
tensor([], size=(1, 0))
tensor([], size=(1, 0))
tensor([], size=(1569, 0))
tensor([], size=(1, 0))
tensor([], size=(10, 0))
tensor([], size=(1438, 0))
tensor([], size=(1, 0))


tensor([], size=(6, 0))
tensor([], size=(4134, 0))
tensor([], size=(1, 0))
tensor([], size=(4, 0))
tensor([], size=(128, 0))
tensor([], size=(1, 0))
tensor([], size=(3, 0))
tensor([], size=(191, 0))
tensor([], size=(1, 0))
tensor([], size=(2, 0))
tensor([], size=(445, 0))
tensor([], size=(1, 0))
tensor([], size=(3, 0))
tensor([], size=(4340, 0))
tensor([], size=(1, 0))
tensor([], size=(3, 0))
tensor([], size=(404, 0))
tensor([], size=(1, 0))
tensor([], size=(6, 0))


: 

: 